In [168]:
import pandas as pd
from sklearn.linear_model import ridge_regression
from pathlib import Path
import numpy as np
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.optimize import linear_sum_assignment

In [169]:


FIXED_PATH = "./embeddings.npy"
TASKS = {
    1: "./subtask1.npy",
    2: "./subtask2.npy",
}
N = 4000
OUT_SUBMISSION = "./submission.csv"

M1 = np.load(FIXED_PATH)

In [170]:
M1.shape

(4000, 384)

In [171]:
M2 = np.load(TASKS[1])
M3 = np.load(TASKS[2])

In [172]:
M2.shape

(4000, 768)

In [173]:
from sklearn.neural_network import MLPRegressor

model1 = MLPRegressor(hidden_layer_sizes=(1000), tol=1e-8).fit(M1[:400], M2[:400])


In [174]:
NewM12 = model1.predict(M1)

In [175]:
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

sims1 = cosine_distances(NewM12, M2)


In [176]:
from scipy.optimize import linear_sum_assignment

In [177]:
pred1 = linear_sum_assignment(sims1)


In [178]:
pred1 = pred1[1]

In [179]:
M2_shuffled = M2[pred1]

M1 -> M2
M2 -> M3
M3 -> M2

M3 -> M1
M1 -> M2

In [ ]:


def fit_map(X, Y):
    model = Ridge(alpha=0.1)
    model.fit(X, Y)
    return model


def solve_subtask2(M1, M2, M3, r2):
    M1n = normalize(M1)
    M2n = normalize(M2)
    M3n = normalize(M3)

    M2_aligned = M2n[r2]

    anchors = np.arange(20)

    A13 = fit_map(M1n[anchors], M3n[anchors])
    A31 = fit_map(M3n[anchors], M1n[anchors])
    A23 = fit_map(M2_aligned[anchors], M3n[anchors])
    A32 = fit_map(M3n[anchors], M2_aligned[anchors])

    M1_to_3 = normalize(A13.predict(M1n))
    M3_to_1 = normalize(A31.predict(M3n))

    M2_to_3 = normalize(A23.predict(M2_aligned))
    M3_to_2 = normalize(A32.predict(M3n))

    S = (
        euclidean_distances(M1_to_3, M3n) +
        euclidean_distances(M1n, M3_to_1) +
        euclidean_distances(M2_to_3, M3n) +
        euclidean_distances(M2_aligned, M3_to_2)
    )
    
    row_ind, col_ind = linear_sum_assignment(S)

    r3 = np.empty(4000, dtype=np.int64)
    r3[row_ind] = col_ind

    return r3

In [181]:
pred2 = solve_subtask2(M1, M2, M3, pred1)

In [182]:
pred2

array([   0,    1,    2, ..., 3670, 1055,  135])

In [183]:
f = open("ans.csv", "w")

f.write("subtaskID,datapointID,answer\n")

for i in range(N):
    f.write(f"{1},{i},{pred1[i]}\n")
for i in range(N):
    f.write(f"{2},{i},{pred2[i]}\n")

f.close()
print(f"Wrote {OUT_SUBMISSION}")


Wrote ./submission.csv


In [184]:
print(pred1[:30])
print(pred2[:30])

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
[   0    1    2    3    4    5    6    7    8    9   10   11   12   13
   14   15   16   17   18   19 2267 3489   60 1103  939 2189  727 1492
  414 2118]
